# Pandas - Datums- und Zeitreihen

Einstieg in Datums- und Zeitreihen mit Pandas

Kennenlernen von
- [`pd.to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)-Funktion
- [`.dt.day_name()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.day_name.html)-Methode
- [`.dt.month_name()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.month_name.html)-Methode
- [`.resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)-Methode 
- [`.asfreq()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.asfreq.html)-Methode

2026-05-13 - ug 1.3

---

Referenzen

Vorbereitung

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## DataFrame erzeugen

+ Die [`.date_range()`](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html)-Funktion erzeugt einen (Zeit-)Index mit Zeitpunkten.
+ Die [`.strftime()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.strftime.html)-Methode ("string format time") wandelt die Zeitpunkte in Zeichenketten mit angegebenen Format um.


In [ ]:

# Zeitreihe erstellen

# Teil 1: Daten erstellen
Datum = pd.date_range("2021-01-01","2021-04-30")   # 4 Monate
Datum = Datum.strftime("%d%m%Y")      # wir wollen Datum hier zu Übungszwecken als eine Zeichenkette haben

Linear = np.arange(len(Datum))        # linear ansteigende Zahlenfolge

np.random.seed(7)
Random = np.random.randn(len(Datum))  # Zufallszahlen
Random_walk = Random.cumsum()         # Zufallszahle aufsummiert

data = {'Datum'     : Datum,
        'Linear'    : Linear, 
        'Random'    : Random,
        'RandomWalk': Random_walk}

# Teil 2: Daten in einem DataFrame speichern
df = pd.DataFrame(data = data)
print(df.shape)
df.head()

In [ ]:
df.info()

Die Spalte `Datum` ist vom Typ `str`. Wir wollen sie aber als ein Datum-Zeit-Typen haben

### Datum mit `pd.to_datetime()`- Funktion umwandeln

Die Funktion [`pd.to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
wandelt das Argument in eine Reihe von Zeitpunkten um.

In [ ]:
# Achtung Anweisung funktioniert nicht, da Datum als Zeichenkette vorliegt
df['Datum'] = pd.to_datetime(df['Datum'])


Das einfache Anwenden der `pd.to_datetime()`- Funktion auf die Spalte `df['Datum']` führt hier zu folgendem Fehler:
```
_ParserError: month must be in 1..12: 01012021_ hervor.
```

Um das Datum in den Zeichenketten korrekt zu interpretieren, benötigen wir einen [Format-String](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior).
<br>
Diesen geben wir mit dem Parameter `format=` an.


|Format|Bedeutung|Beispiel|
| ---  | --- | --- |
|  %Y  | Jahreszahl vierstellig mit vorangestellten Nullen | 0001, 0002, …, 2013, 2014, …, 9998, 9999|
|  %y  | Jahreszahl zweistellig mit vorangestellter Null   | 00, 01, …, 99|
|  %m  | Monat als Zahl mit vorangestellter Null| 01, 02, …, 12|
|  %d  | Tag als Zahl mit vorangestellter Null | 01, 02, …, 31 |
|  %H  | Stunden (24h-Uhr) als Zahl mit vorangestellter Null |00, 01, …, 23|
|  %M  | Minuten als Zahl mit vorangesteller Null | 00, 01, …, 59|
|  %S  | Sekunden als Zahl mit vornagesteller Null | 00, 01, …, 59|
|  %f  | Mikrosekunden | 000000, 000001, …, 999999|


Das Datum in der Spalte `df['Datum']` ist im Format Tag, Monat und Jahr z.B. _01012021_ abgelegt.

Es wird der Format-String `format="%d%m%Y"` benötigt.

In [ ]:
df['Datum'] = pd.to_datetime(df['Datum'],format="%d%m%Y")
df.info()

Jetzt ist die Spalte `df['Datum']` vom Datentype `datetime64[ns]`.

### Zeitzugriff-Objekt

Über das Zeitzugriff-Objekt `.dt` können das Jahr (`.year`), der Monat (`.month`) und der Tag (`.day`) als Pandas Series-Objekt zugegriffen werden:

In [ ]:
df['Datum'].dt.year

In [ ]:
df['Datum'].dt.month

In [ ]:
df['Datum'].dt.day

### Monat und Tag als Name wandeln

-  [`.dt.day_name()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.day_name.html)
-  [`.dt.month_name()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.month_name.html)


In [ ]:
df['Datum'].dt.day_name()

In [ ]:
df['Datum'].dt.month_name()

### das früheste und das späteste Datum ermitteln

Das früheste Datum bekommen wir über die `.min()`-Methode als `Timestamp`-Objekt zurückgeliefert.

In [ ]:
df['Datum'].min()

Das späteste Datum bekommen wir über die `.max()`-Methode als `Timestamp`-Objekt zurückgeliefert.

In [ ]:
df['Datum'].max()

### Ausrechnen einer Zeitspanne

Die Differenz zweier `Timestamp`-Objekte liefert ein `Timedelta`-Objekt.

In [ ]:
df['Datum'].max()-df['Datum'].min()

### Filtern von Reihen mit Zeitpunkten

a) Vergleich mit Zeichenketten

Beispiel: Alle Zeitpunkte nach dem 1. Februar 2021. (Die -01 für den Tag kann weggelassen werden)

In [ ]:
mask = df['Datum'] >= '2021-02'
df[mask]

b) unter Verwendung der `pd.to_datetime()` Funktion

Hier lassen sich komplexere Bedingungen bauen:

Beispiel: Zeitpunkte zwischen dem 2. und 10. Februar

In [ ]:
mask = (df['Datum'] >= pd.to_datetime('2021-02-02')) & (df['Datum'] <= pd.to_datetime('2021-02-10'))
df[mask]

### Datum als Index setzen

In [ ]:
df2 = df.set_index('Datum')
df2.head()

#### Zugriff und Filtern über die `.loc[]`-Methode 

Indexing

In [ ]:
df2.loc['2021-03']

Slicing

Anfang und Ende sind enthalten !!!

In [ ]:
df2.loc['2021-02-02':'2021-02-10']

Slicing und nur Rückgabe der Spalte 'Random'

In [ ]:
df2.loc['2021-02-02':'2021-02-10','Random']

### Anwenden einer Aggregationsmethode

Durch die Aggregationsmethode Mittelwert `.mean()` werden die Wert einer Spalte zu einem Wert zusammen gefasst.

Aggregationsmethoden fassen viele Wert zu einem Wert zusammen.

In [ ]:
df2['Random'].mean()

#### Resampling 

Mit der [`.resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)-Methode 
lassen sich Wert nach einem bestimmten Muster zusammenfassen (aggregieren).

Beispiel: Für Werte wochenweise als Mittelwert zusammenfassen.

- Auswahl der Wert über Angabe der Spalte: `df2['RandomWalk']`
- Bilden der wochenweisen Abtastung: `.resample('W')`
- Anwenden der Aggregationsmethode: `.mean()`

In [ ]:
df3 = df2['RandomWalk'].resample('W').mean()
df3

Wochentage überprüfen -> alle Sonntage

In [ ]:
df3.index.to_series().dt.day_name() 

#### Plot von Spalte 'RandomWalk' und den gesampelten 'RandomWalk' Werten

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

df2['RandomWalk'].plot(ax=ax,marker='o',label='RandomWalk')
df3.plot(ax=ax,marker='o',label='RandomWalk Sampled')
ax.grid(True)
ax.legend()

### `.asfreq()`-Methode 
Mit der [`.asfreq()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.asfreq.html)-Methode formen wir die Serie `df3` wieder eine Serie mit täglichem Raster um.
<br>
Warning:
<br>
Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df3.asfreq('1d').head(15)




In [ ]:
df3.asfreq('1D').head(15)

Jetzt fehlen aber die Wert an den Tagen zwischen den Sonntagen.

Mit dem Parameter `method='ffill'` können diese aufgefüllt werden.

In [ ]:
df3.asfreq('1D',method='ffill').head(17)